In [ ]:
!pip install -q -U langchain langchain-google-genai
!pip install -q pandas==2.2.2

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.7/12.7 MB 74.4 MB/s eta 0:00:00


In [ ]:
import pandas as pd
print(pd.__version__)

2.2.2


In [5]:
from google.colab import userdata
import os

os.environ["GOOGLE_API_KEY"] = userdata.get("GEMINI_API_KEY")

print("API key loaded:", os.environ["GOOGLE_API_KEY"] is not None)

API key loaded: True


In [19]:
import pandas as pd

exercise_data = [
    ["Push-Up", "Chest/Triceps", "Bodyweight", "Beginner", "general fitness", "3", "10-15"],
    ["Bodyweight Squat", "Legs", "Bodyweight", "Beginner", "fat loss", "3", "12-15"],
    ["Walking Lunges", "Legs", "Bodyweight", "Beginner", "fat loss", "3", "10 each leg"],
    ["Plank", "Core", "Bodyweight", "Beginner", "general fitness", "3", "30-45 sec"],
    ["Jumping Jacks", "Full Body", "Bodyweight", "Beginner", "fat loss", "3", "30 sec"],
    ["Glute Bridge", "Glutes", "Bodyweight", "Beginner", "general fitness", "3", "12-15"],

    ["Goblet Squat", "Legs", "Dumbbell", "Beginner", "fat loss", "3", "10-12"],
    ["Dumbbell Lunges", "Legs", "Dumbbell", "Beginner", "fat loss", "3", "10 each leg"],
    ["Dumbbell Romanian Deadlift", "Hamstrings/Glutes", "Dumbbell", "Beginner", "fat loss", "3", "10-12"],
    ["Dumbbell Step-Up", "Legs/Glutes", "Dumbbell", "Beginner", "fat loss", "3", "10 each leg"],

    ["Dumbbell Row", "Back", "Dumbbell", "Beginner", "muscle gain", "3", "8-12"],
    ["Dumbbell Shoulder Press", "Shoulders", "Dumbbell", "Beginner", "muscle gain", "3", "8-12"],
    ["Dumbbell Chest Press", "Chest", "Dumbbell", "Beginner", "muscle gain", "3", "8-12"],
    ["Dumbbell Bicep Curl", "Arms", "Dumbbell", "Beginner", "muscle gain", "3", "10-12"],
    ["Dumbbell Triceps Extension", "Arms", "Dumbbell", "Beginner", "muscle gain", "3", "10-12"],

    ["Dumbbell Farmer Carry", "Full Body/Core", "Dumbbell", "Beginner", "general fitness", "3", "30 sec"],
    ["Dumbbell Deadlift", "Back/Legs", "Dumbbell", "Beginner", "general fitness", "3", "10-12"],
    ["Standing Calf Raise", "Calves", "Bodyweight", "Beginner", "general fitness", "3", "12-20"],
]

exercise_df = pd.DataFrame(
    exercise_data,
    columns=[
        "exercise_name",
        "muscle_group",
        "equipment",
        "difficulty",
        "goal_type",
        "sets",
        "reps"
    ]
)

exercise_df

,exercise_name,muscle_group,equipment,difficulty,goal_type,sets,reps
0,Push-Up,Chest/Triceps,Bodyweight,Beginner,general fitness,3,10-15
1,Bodyweight Squat,Legs,Bodyweight,Beginner,fat loss,3,12-15
2,Walking Lunges,Legs,Bodyweight,Beginner,fat loss,3,10 each leg
3,Plank,Core,Bodyweight,Beginner,general fitness,3,30-45 sec
4,Jumping Jacks,Full Body,Bodyweight,Beginner,fat loss,3,30 sec
5,Glute Bridge,Glutes,Bodyweight,Beginner,general fitness,3,12-15
6,Goblet Squat,Legs,Dumbbell,Beginner,fat loss,3,10-12
7,Dumbbell Lunges,Legs,Dumbbell,Beginner,fat loss,3,10 each leg
8,Dumbbell Romanian Deadlift,Hamstrings/Glutes,Dumbbell,Beginner,fat loss,3,10-12
9,Dumbbell Step-Up,Legs/Glutes,Dumbbell,Beginner,fat loss,3,10 each leg


In [20]:
food_db = {
    "chicken breast": {"calories": 165, "protein": 31, "carbs": 0, "fat": 3.6},
    "rice": {"calories": 130, "protein": 2.7, "carbs": 28, "fat": 0.3},
    "egg": {"calories": 78, "protein": 6, "carbs": 0.6, "fat": 5},
    "oats": {"calories": 150, "protein": 5, "carbs": 27, "fat": 3},
    "salmon": {"calories": 208, "protein": 20, "carbs": 0, "fat": 13},
    "banana": {"calories": 105, "protein": 1.3, "carbs": 27, "fat": 0.4},
    "greek yogurt": {"calories": 100, "protein": 10, "carbs": 4, "fat": 0.7},
}

def classify_user_profile(goal, equipment, difficulty="Beginner"):
    goal_lower = goal.lower()
    equipment_lower = equipment.lower()

    if "fat" in goal_lower or "lose" in goal_lower or "weight loss" in goal_lower:
        plan_type = "fat loss"
    elif "muscle" in goal_lower or "bulk" in goal_lower or "gain" in goal_lower:
        plan_type = "muscle gain"
    else:
        plan_type = "general fitness"

    if "dumbbell" in equipment_lower:
        equipment_class = "Dumbbell"
    elif "bodyweight" in equipment_lower or "no equipment" in equipment_lower:
        equipment_class = "Bodyweight"
    else:
        equipment_class = "Bodyweight"

    return {
        "plan_type": plan_type,
        "equipment_class": equipment_class,
        "difficulty": difficulty
    }

def find_exercises(goal_type, equipment, difficulty="Beginner"):
    equipment = equipment.lower()

    if "dumbbell" in equipment:
        equipment = "Dumbbell"
    elif "bodyweight" in equipment:
        equipment = "Bodyweight"

    results = exercise_df[
        (exercise_df["goal_type"].str.contains(goal_type, case=False, na=False)) &
        (exercise_df["equipment"].str.contains(equipment, case=False, na=False)) &
        (exercise_df["difficulty"].str.contains(difficulty, case=False, na=False))
    ]

    if results.empty:
        results = exercise_df[
            (exercise_df["equipment"].str.contains(equipment, case=False, na=False))
        ]

    return results.to_dict(orient="records")

def get_food_info(food_name):
    food_name = food_name.lower().strip()
    return food_db.get(food_name, {"error": "Food not found in small demo database."})

def estimate_macros(weight_lbs, goal):
    goal = goal.lower()
    protein = round(weight_lbs * 0.8)

    if "fat" in goal or "lose" in goal:
        calories = round(weight_lbs * 12)
    elif "muscle" in goal or "gain" in goal:
        calories = round(weight_lbs * 16)
    else:
        calories = round(weight_lbs * 14)

    return {
        "estimated_daily_calories": calories,
        "estimated_daily_protein_grams": protein
    }

print("Nutrition data and helper functions loaded successfully.")

Nutrition data and helper functions loaded successfully.


In [15]:
test_profile = classify_user_profile(
    goal="I want to lose fat",
    equipment="dumbbells",
    difficulty="Beginner"
)

print("Classified Profile:")
print(test_profile)

print("\nExercise Results:")
print(find_exercises(test_profile["plan_type"], test_profile["equipment_class"]))

print("\nMacro Estimate:")
print(estimate_macros(180, test_profile["plan_type"]))

print("\nFood Lookup:")
print(get_food_info("chicken breast"))

Classified Profile:
{'plan_type': 'fat loss', 'equipment_class': 'Dumbbell', 'difficulty': 'Beginner'}

Exercise Results:
[{'exercise_name': 'Goblet Squat', 'muscle_group': 'Legs', 'equipment': 'Dumbbell', 'difficulty': 'Beginner', 'goal_type': 'fat loss', 'sets': '3', 'reps': '10-12'}]

Macro Estimate:
{'estimated_daily_calories': 2160, 'estimated_daily_protein_grams': 144}

Food Lookup:
{'calories': 165, 'protein': 31, 'carbs': 0, 'fat': 3.6}


In [16]:
from langchain.tools import tool

@tool
def lookup_exercises(goal_type: str, equipment: str, difficulty: str = "Beginner") -> str:
    """
    Finds exercises that match the user's fitness goal, available equipment, and difficulty level.
    """
    results = find_exercises(goal_type, equipment, difficulty)
    return str(results)


@tool
def nutrition_lookup(food_name: str) -> str:
    """
    Looks up basic nutrition information for a food item.
    """
    result = get_food_info(food_name)
    return str(result)


@tool
def macro_estimator(weight_lbs: float, goal: str) -> str:
    """
    Estimates daily calories and protein grams based on user weight and fitness goal.
    """
    result = estimate_macros(weight_lbs, goal)
    return str(result)


tools = [lookup_exercises, nutrition_lookup, macro_estimator]

print("LangChain tools created successfully.")

LangChain tools created successfully.


In [17]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.agents import create_agent

model = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0.4
)

agent = create_agent(
    model=model,
    tools=tools,
    system_prompt="""
You are FitGuide, an AI fitness and nutrition coach agent.

Your job:
1. Understand the user's fitness goal.
2. Use tools when needed.
3. Create a simple beginner-friendly plan.
4. Keep advice safe and general.
5. Do not claim to be a doctor, dietitian, or certified trainer.

Always respond using this format:

1. Goal Summary
2. Workout Recommendation
3. Nutrition Guidance
4. Estimated Calories and Protein
5. Weekly Coaching Tip
"""
)

print("FitGuide agent recreated successfully with Gemini 2.5 Flash.")

FitGuide agent recreated successfully with Gemini 2.5 Flash.


In [18]:
user_goal = "I want to lose fat"
user_equipment = "dumbbells"
user_difficulty = "Beginner"
user_weight = 180

classified_profile = classify_user_profile(
    goal=user_goal,
    equipment=user_equipment,
    difficulty=user_difficulty
)

print("DEEP LEARNING / PROFILE CLASSIFIER OUTPUT:")
print(classified_profile)

prompt = f"""
User goal: {user_goal}
Available equipment: {user_equipment}
Difficulty level: {user_difficulty}
Weight: {user_weight} lbs

Classified user profile:
{classified_profile}

Please create a personalized fitness and nutrition plan.
Use the tools if needed.
"""

response = agent.invoke({
    "messages": [
        {"role": "user", "content": prompt}
    ]
})

print("\nFITGUIDE AGENT RESPONSE:")
print(response)

DEEP LEARNING / PROFILE CLASSIFIER OUTPUT:
{'plan_type': 'fat loss', 'equipment_class': 'Dumbbell', 'difficulty': 'Beginner'}

FITGUIDE AGENT RESPONSE:
{'messages': [HumanMessage(content="\nUser goal: I want to lose fat\nAvailable equipment: dumbbells\nDifficulty level: Beginner\nWeight: 180 lbs\n\nClassified user profile:\n{'plan_type': 'fat loss', 'equipment_class': 'Dumbbell', 'difficulty': 'Beginner'}\n\nPlease create a personalized fitness and nutrition plan.\nUse the tools if needed.\n", additional_kwargs={}, response_metadata={}, id='c2bf31d9-c3c7-41f3-8d43-fd00c89cbc78'), AIMessage(content='', additional_kwargs={'function_call': {'name': 'macro_estimator', 'arguments': '{"weight_lbs": 180, "goal": "fat loss"}'}, '__gemini_function_call_thought_signatures__': {'34bda1d7-d5af-4c5f-8f0a-27c48b44f644': 'CusFAQw51scAqJzpcbN02uS9cVVaqq4Nbrd/3Z0MyqdB4RXY1s0sv371OwET+uthhf1o4Vw/fBBhBBRJdtVyCDG34OvnZtHlJTCJ+1Np2bgbm01cDL774Xvm9a0sBhUSbaBI8lJCVYvdH8hkPBft4q6UlNHz3iJIgioNsyaKTnXcIymdPbSMW

In [21]:
# Simple memory store for FitGuide
# This keeps track of user progress notes during the session.

progress_memory = []

def save_progress(user_name, note):
    progress_memory.append({
        "user_name": user_name,
        "note": note
    })
    return f"Progress saved for {user_name}: {note}"

def view_progress(user_name):
    user_notes = [
        item["note"] for item in progress_memory
        if item["user_name"].lower() == user_name.lower()
    ]

    if not user_notes:
        return f"No progress notes found for {user_name}."

    return user_notes

print("Memory functions loaded successfully.")

Memory functions loaded successfully.


In [22]:
from langchain.tools import tool

@tool
def save_progress_note(user_name: str, note: str) -> str:
    """
    Saves a user's fitness or nutrition progress note into memory.
    Use this when the user reports progress, completed workouts, weight changes, or consistency updates.
    """
    return save_progress(user_name, note)


@tool
def view_progress_notes(user_name: str) -> str:
    """
    Retrieves saved progress notes for a user.
    Use this when creating updated recommendations based on past progress.
    """
    return str(view_progress(user_name))


# Rebuild tools list with memory tools included
tools = [
    lookup_exercises,
    nutrition_lookup,
    macro_estimator,
    save_progress_note,
    view_progress_notes
]

print("Memory tools added successfully.")
print("Total tools:", len(tools))

Memory tools added successfully.
Total tools: 5


In [23]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.agents import create_agent

model = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0.4
)

agent = create_agent(
    model=model,
    tools=tools,
    system_prompt="""
You are FitGuide, an AI fitness and nutrition coach agent.

You have access to tools for:
- looking up exercises
- looking up nutrition information
- estimating calories and protein
- saving user progress notes
- viewing user progress notes

Use tools whenever they help answer the user's request.

Important behavior:
1. If the user gives a new fitness goal, create a simple beginner-friendly plan.
2. If the user reports progress, save it using the memory tool.
3. If the user asks for an updated plan, check memory first.
4. Keep advice safe and general.
5. Do not claim to be a doctor, dietitian, or certified trainer.

Always respond using this format when creating a plan:

1. Goal Summary
2. Workout Recommendation
3. Nutrition Guidance
4. Estimated Calories and Protein
5. Weekly Coaching Tip
"""
)

print("FitGuide agent recreated successfully with memory tools.")

FitGuide agent recreated successfully with memory tools.


In [24]:
response = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "My name is Alex. I completed 2 dumbbell workouts this week and stayed close to my calorie goal."
        }
    ]
})

print(response["messages"][-1].content)

Great job, Alex! It's fantastic to hear you completed 2 dumbbell workouts and stayed on track with your calorie goals this week. Consistency is key, and you're doing great! Keep up the excellent work!


In [25]:
print(progress_memory)

[{'user_name': 'Alex', 'note': 'Completed 2 dumbbell workouts this week and stayed close to calorie goal.'}]


In [26]:
response = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "My name is Alex. Based on my progress, give me an updated fat loss plan."
        }
    ]
})

print(response["messages"][-1].content)

[{'type': 'text', 'text': "Here's an updated fat loss plan for you, Alex, based on your progress of completing 2 dumbbell workouts and staying close to your calorie goal!\n\n1. Goal Summary\nContinue with your fat loss journey by focusing on consistent workouts and mindful eating to create a sustainable calorie deficit.\n\n2. Workout Recommendation\nSince you've been consistent with dumbbell workouts, let's build on that. Aim for 3-4 sessions per week, focusing on full-body movements. Here are some ideas for dumbbell exercises:\n*   **Dumbbell Squats:** 3 sets of 10-12 reps\n*   **Dumbbell Rows:** 3 sets of 10-12 reps per arm\n*   **Dumbbell Chest Press (on floor or bench):** 3 sets of 10-12 reps\n*   **Dumbbell Lunges:** 3 sets of 10-12 reps per leg\n*   **Dumbbell Shoulder Press:** 3 sets of 10-12 reps\n\nRemember to choose a weight that challenges you while allowing you to maintain good form.\n\n3. Nutrition Guidance\nContinue to focus on whole, unprocessed foods. Prioritize lean pr

In [27]:
def print_final_response(response):
    final_message = response["messages"][-1].content

    if isinstance(final_message, list):
        for item in final_message:
            if isinstance(item, dict) and item.get("type") == "text":
                print(item.get("text"))
    else:
        print(final_message)

print("Clean response printer ready.")

Clean response printer ready.


In [28]:
print_final_response(response)

Here's an updated fat loss plan for you, Alex, based on your progress of completing 2 dumbbell workouts and staying close to your calorie goal!

1. Goal Summary
Continue with your fat loss journey by focusing on consistent workouts and mindful eating to create a sustainable calorie deficit.

2. Workout Recommendation
Since you've been consistent with dumbbell workouts, let's build on that. Aim for 3-4 sessions per week, focusing on full-body movements. Here are some ideas for dumbbell exercises:
*   **Dumbbell Squats:** 3 sets of 10-12 reps
*   **Dumbbell Rows:** 3 sets of 10-12 reps per arm
*   **Dumbbell Chest Press (on floor or bench):** 3 sets of 10-12 reps
*   **Dumbbell Lunges:** 3 sets of 10-12 reps per leg
*   **Dumbbell Shoulder Press:** 3 sets of 10-12 reps

Remember to choose a weight that challenges you while allowing you to maintain good form.

3. Nutrition Guidance
Continue to focus on whole, unprocessed foods. Prioritize lean protein, plenty of vegetables, and healthy fa

In [29]:
user_goal = "I want to lose fat"
user_equipment = "dumbbells"
user_difficulty = "Beginner"
user_weight = 180

classified_profile = classify_user_profile(user_goal, user_equipment, user_difficulty)

prompt = f"""
User goal: {user_goal}
Available equipment: {user_equipment}
Difficulty level: {user_difficulty}
Weight: {user_weight} lbs

Classified user profile:
{classified_profile}

Please create a personalized fitness and nutrition plan.
Use the tools if needed.
"""

response = agent.invoke({"messages": [{"role": "user", "content": prompt}]})
print_final_response(response)

Here is your personalized fitness and nutrition plan to help you achieve your fat loss goal:

### 1. Goal Summary
Your primary goal is fat loss. This plan focuses on creating a calorie deficit through a combination of effective dumbbell exercises and mindful eating, suitable for a beginner level.

### 2. Workout Recommendation
Perform this full-body dumbbell workout 2-3 times per week on non-consecutive days. Focus on proper form and controlled movements.

*   **Goblet Squat:** 3 sets of 10-12 repetitions
*   **Dumbbell Lunges:** 3 sets of 10 repetitions per leg
*   **Dumbbell Romanian Deadlift:** 3 sets of 10-12 repetitions
*   **Dumbbell Step-Up:** 3 sets of 10 repetitions per leg (use a sturdy chair or bench)

Remember to warm up for 5-10 minutes before your workout with light cardio and dynamic stretches, and cool down with static stretches afterward.

### 3. Nutrition Guidance
To support fat loss, focus on a balanced diet rich in whole foods.
*   **Prioritize Protein:** Include le

In [30]:
response = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "My name is Alex. I completed 2 dumbbell workouts this week and stayed close to my calorie goal."
        }
    ]
})

print_final_response(response)

That's great to hear, Alex! Consistency is key, and completing your workouts and sticking to your calorie goals are fantastic progress. Keep up the excellent work!


In [31]:
response = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "My name is Alex. Based on my progress, give me an updated fat loss plan."
        }
    ]
})

print_final_response(response)

Here is an updated fat loss plan for you, Alex, based on your consistent progress with dumbbell workouts and calorie goals!

### 1. Goal Summary
Continue your journey towards fat loss through consistent exercise and mindful nutrition.

### 2. Workout Recommendation
Since you've been consistent with 2 dumbbell workouts a week, let's aim for 3 days a week to gently increase your activity. Focus on proper form and controlled movements.

**Full Body Dumbbell Workout (Perform 3 times a week, with at least one rest day in between):**
*   **Goblet Squat:** 3 sets of 10-12 reps
*   **Dumbbell Lunges:** 3 sets of 10 reps each leg
*   **Dumbbell Romanian Deadlift:** 3 sets of 10-12 reps
*   **Dumbbell Step-Up:** 3 sets of 10 reps each leg (use a sturdy chair or bench)

Remember to warm up for 5-10 minutes with light cardio and dynamic stretches before your workout, and cool down with static stretches afterward.

### 3. Nutrition Guidance
Continue to focus on a balanced diet to support fat loss. 